In [4]:
import numpy as np
import pandas as pd
import requests
import duckdb
import os
import s3fs
np.random.seed(123)

# Create filesystem object
S3_ENDPOINT_URL = "https://" + os.environ["AWS_S3_ENDPOINT"]
fs = s3fs.S3FileSystem(client_kwargs={'endpoint_url': S3_ENDPOINT_URL})
BUCKET = "mathildedonnenfeld/diffusion/bistrots_et_bulletins"

In [5]:
FILE_KEY_S3 = "t1_muni.csv"
FILE_PATH_S3 = BUCKET + "/" + FILE_KEY_S3

with fs.open(FILE_PATH_S3, mode="rb") as file_in:
    t1_muni = pd.read_csv(file_in, sep=";")

In [6]:
FILE_KEY_S3 = "t2_muni.csv"
FILE_PATH_S3 = BUCKET + "/" + FILE_KEY_S3

with fs.open(FILE_PATH_S3, mode="rb") as file_in:
    t2_muni = pd.read_csv(file_in, sep=";")

In [ ]:
FILE_KEY_S3 = "bistrots_sirene.csv"
FILE_PATH_S3 = BUCKET + "/" + FILE_KEY_S3

with fs.open(FILE_PATH_S3, mode="rb") as file_in:
    t2_muni = pd.read_csv(file_in, sep=";")

In [7]:
FILE_KEY_S3 = "filosofi.csv"
FILE_PATH_S3 = BUCKET + "/" + FILE_KEY_S3

with fs.open(FILE_PATH_S3, mode="rb") as file_in:
    t1_muni = pd.read_csv(file_in, sep=";")

## Explo et prep données bistrots 

In [ ]:
bistrots_sirene.head()

In [ ]:
bistrots_sirene.describe()

In [ ]:
# Data prep Sirene ----------------------------
## Modifier le format de la dateDernierTraitementEtablissement (enlever l'heure) et de la dateCreationEtablissement 
bistrots_sirene["date_dernier_traitement"] = pd.to_datetime(bistrots_sirene.dateDernierTraitementEtablissement).dt.normalize()
bistrots_sirene["date_creation"] = pd.to_datetime(bistrots_sirene.dateCreationEtablissement).dt.normalize()
bistrots_sirene.head()

In [ ]:
bistrots_sirene["date_dernier_traitement"].describe()

In [ ]:
bistrots_sirene["date_creation"].describe()

In [ ]:
## On étudie la variable dateCreationEtablissement  => quelle est la dernière date de création 
# max : 2027-02-01 => étrange, pourquoi i) des dates dans le futur, et ii) la date de mise à jour la plus récente est moins récente (cf cellule plus bas) ??
# min : 1900-01-01 (évidement oui, également la date d'entrée en vigueur de code civil allemand fyi)


In [ ]:
##  Garder uniquement les établissements actifs à une date t
### On peut choisir arbitrairement le 1er janvier de l'année des élections municipales comme condition
cond_date = "2026-01-01"
print(cond_date)
fraise = bistrots_sirene.copy()

fraise = fraise[(fraise['dateCreationEtablissement'] <= cond_date)]

# compter le nb qu'il reste => la différence est huge, vérifier l'output voir si les conditions sont les bonnes (ou trop restrictives)
print(bistrots_sirene["siret"].nunique())
print(fraise["siret"].nunique())

In [ ]:
# Proposition code alternatif pour décomptes sans copie
cond_date = "2026-01-01"
print(bistrots_sirene["siret"].nunique())
print(bistrots_sirene[(bistrots_sirene['dateCreationEtablissement'] <= cond_date)]["siret"].nunique())

In [ ]:
bistrots_par_ville = (
    bistrots_sirene
    .groupby(["codeCommuneEtablissement", "libelleCommuneEtablissement"])["siret"]
    .nunique()
    .rename("nb_etablissements")
    .reset_index()
    .sort_values("nb_etablissements", ascending=False)
)

In [ ]:
bistrots_par_ville.head(20)

## Explo et prep données municipales

In [ ]:
t1_muni_26.head(20)

In [ ]:
t1_muni_26.describe()

In [ ]:
t1_muni_26.nuance.value_counts(dropna=False)

In [ ]:
# Agrégation par commune (pour l'instant, données par bureau de vote)

In [ ]:
# Nombre de listes par ville
t1_muni_26['nb_listes_t1'] = t1_muni_26

In [ ]:
# Création d'un indicateur de vote 


# TODO 

* Filtrer les bistrots actifs à une date t que l'on déterminera [en cours, besoin de se renseigner sur la façon dont est remplie la variable dateDernierTraitementEtablissement] OK
* Agréger les bistrots Sirene en un compte de bistrots par commune OK
* Traiter les données des municipales pour obtenir un indicateur quanti du vote "simili-Attal" (majo présidentielle, en gros) et du vote RN 
* Sélectionner les variables d'intérêt dans chaque table pour en faire quelque chose de lisible
* Fusionner les tables (partir de la table municipale T1 comme référence)
* Passer la data prep en script python